# Aula 1.2 — Slicing e Operações em Imagens

**Bloco 1 — NumPy**  
**Projeto:** IC-FAPEMA — Detecção de capacetes em motociclistas via drone  
**Objetivo:** Aprender a fatiar arrays de imagem para extrair canais, recortes e regiões de detecção

---

## Termos

| Termo | Conceito | Função no código |
|---|---|---|
| **Slicing** | Fatiar — pegar um pedaço do array | `frame[100:200, 300:400]` |
| **Index** | Acessar uma posição específica | `frame[100, 200]` |
| **Crop** | Recorte de uma região da imagem | `frame[y1:y2, x1:x2]` |
| **Channel** | Canal de cor — R, G ou B | `frame[:, :, 0]` |
| **ROI** | Região de Interesse — onde está o objeto | `frame[y1:y2, x1:x2]` |

---

## Conceito

A sintaxe de slicing é sempre:
```
imagem[linhas, colunas, canais]
```
O `:` significa **tudo**. Exemplos:
- `frame[:, :, 0]` → todas as linhas, todas as colunas, só canal R
- `frame[100:300, 200:450]` → recorte entre linhas 100-300 e colunas 200-450

---

## Bounding Box — de onde vêm os valores?

O modelo YOLO retorna automaticamente:
```
[classe, x_centro, y_centro, largura, altura]
```
Todos entre 0 e 1 (proporcionais ao tamanho da imagem).  
Você converte para pixels com a fórmula padrão:
```
x1 = (x_centro - largura/2) × largura_frame
x2 = (x_centro + largura/2) × largura_frame
y1 = (y_centro - altura/2)  × altura_frame
y2 = (y_centro + altura/2)  × altura_frame
```

In [ ]:
import numpy as np

# Simula um frame do drone — 480x640 colorido
# shape (480, 640, 3) → altura=480, largura=640, 3 canais RGB
frame = np.random.randint(0, 256, size=(480, 640, 3), dtype=np.uint8)
print("Shape do frame:", frame.shape)

## Parte A — Isolando canais RGB

In [ ]:
# O : significa 'todas as linhas' e 'todas as colunas'
# O número final seleciona o canal: 0=R, 1=G, 2=B
canal_r = frame[:, :, 0]   # canal vermelho
canal_g = frame[:, :, 1]   # canal verde
canal_b = frame[:, :, 2]   # canal azul

print("Canal R:", canal_r.shape)   # (480, 640) — sem a dimensão do canal
print("Canal G:", canal_g.shape)   # (480, 640)
print("Canal B:", canal_b.shape)   # (480, 640)

## Parte B — Recortando uma região (ROI)

In [ ]:
# Recorta a região entre linhas 100-300 e colunas 200-450
# Imagina que o motociclista está nessa área do frame
roi = frame[100:300, 200:450]

print("Shape do frame completo:", frame.shape)   # (480, 640, 3)
print("Shape do recorte (ROI):", roi.shape)      # (200, 250, 3)

## Parte C — Convertendo bounding box YOLO para pixels

In [ ]:
# Dimensões do frame
altura_frame  = 480
largura_frame = 640

# Valores retornados pelo YOLO (simulados aqui)
# Na prática o modelo retorna esses valores automaticamente
x_centro   = 0.52
y_centro   = 0.38
largura_box = 0.20
altura_box  = 0.15

# Fórmula padrão de conversão YOLO → pixels
x1 = int((x_centro - largura_box/2) * largura_frame)  # borda esquerda
y1 = int((y_centro - altura_box /2) * altura_frame)   # borda superior
x2 = int((x_centro + largura_box/2) * largura_frame)  # borda direita
y2 = int((y_centro + altura_box /2) * altura_frame)   # borda inferior

print(f"Coordenadas em pixels: x1={x1}, y1={y1}, x2={x2}, y2={y2}")

# Recorta a região do capacete detectado
capacete = frame[y1:y2, x1:x2]
print("Shape do capacete recortado:", capacete.shape)

## Parte D — Função reutilizável

In [ ]:
# Função que converte bbox YOLO para pixels
# Você escreve uma vez e usa para qualquer detecção
def bbox_para_pixels(x_centro, y_centro, largura_box, altura_box, largura_frame, altura_frame):
    x1 = int((x_centro - largura_box/2) * largura_frame)
    y1 = int((y_centro - altura_box /2) * altura_frame)
    x2 = int((x_centro + largura_box/2) * largura_frame)
    y2 = int((y_centro + altura_box /2) * altura_frame)
    return x1, y1, x2, y2  # retorna as 4 coordenadas

# Usando a função
x1, y1, x2, y2 = bbox_para_pixels(0.52, 0.38, 0.20, 0.15, 640, 480)
print(f"x1={x1}, y1={y1}, x2={x2}, y2={y2}")

capacete = frame[y1:y2, x1:x2]
print("Shape:", capacete.shape)

---
## Resumo da aula

| O que aprendeu | Para que serve no projeto |
|---|---|
| `frame[:, :, 0]` | Isolar canal de cor |
| `frame[y1:y2, x1:x2]` | Recortar região detectada |
| Fórmula bbox YOLO | Converter detecção para pixels |
| Função reutilizável | Usar em qualquer frame do projeto |